# Runnables Deep Dive: Everything Is a Runnable [Step 4 - Runnables deep dive]

> **MLCourse - Agentic AI - LCEL and Runnables**

> Stage in the capstone: the entire RAG pipeline IS one runnable chain; memory
> wrapping uses RunnableWithMessageHistory.

## What you'll learn
- the uniform Runnable protocol: `invoke`, `batch`, `stream` on every component
- assembling `prompt | model | parser` one stage at a time, swapping live and offline pieces
- `RunnableLambda`: wrapping your own functions as first-class chain stages
- `RunnablePassthrough` plus `.assign()`: keeping inputs while ADDING keys (mini RAG shape)
- `RunnableParallel`: fanning one input out to a summary branch and a keywords branch
- resilience and control: `.with_fallbacks()`, `.stream()` tokens, `.batch()`, `bind(stop=...)`

Roughly half this notebook runs with ZERO providers - lambdas, passthroughs, and
parallel fan-outs are pure Python. Live model cells are individually guarded.

In [1]:
# Standard first cell for every MLCourse notebook: imports, inline plotting,
# and automatic discovery of the track-level .env file.
import os                                   # read environment variables such as GROQ_API_KEY
from pathlib import Path                    # walk up the folder tree hunting for .env

try:                                        # Jupyter kernels define get_ipython();
    get_ipython().run_line_magic("matplotlib", "inline")  # render plots inside the notebook
except NameError:                           # plain python runs have no IPython,
    pass                                    # so skip the magic silently

from dotenv import load_dotenv              # loads KEY=VALUE lines into os.environ


def find_track_env(start: Path) -> "Path | None":
    """Climb from *start* upward until 03_agentic_ai/.env appears."""
    for folder in (start, *start.parents):             # current dir, then every parent
        candidate = folder / "03_agentic_ai" / ".env"  # track secrets live at this spot
        if candidate.is_file():                        # hit: stop climbing immediately
            return candidate
    return None                                        # miss everywhere: caller decides


_env_path = find_track_env(Path.cwd())     # search from wherever the kernel started
if _env_path is not None:                  # found the track root?
    load_dotenv(_env_path)                 # push GROQ_API_KEY etc. into os.environ
    print("[setup] loaded env:", _env_path)
else:
    print("[setup] no 03_agentic_ai/.env found - live demos will be skipped")

# Shared imports for the rest of the notebook - all keyless, all local.
from langchain_core.messages import AIMessage               # chat models emit these
from langchain_core.prompts import ChatPromptTemplate       # dict -> rendered messages
from langchain_core.output_parsers import StrOutputParser   # AIMessage -> str
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.runnables import RunnableLambda         # wrap ANY callable
from langchain_core.runnables import RunnablePassthrough    # identity stage with .assign()
from langchain_core.runnables import RunnableParallel       # fan-out / merge branches

[setup] no 03_agentic_ai/.env found - live demos will be skipped


## 1. One protocol to rule them all

Every LangChain component - templates, models, parsers, retrievers, even raw
functions wrapped below - implements the same three verbs:

- `invoke(x)` : one input in, one output out
- `batch([x1, x2])` : many in, list out, ORDER PRESERVED
- `stream(x)` : lazy iterator yielding chunks as each stage finishes

Uniformity is what makes composition possible: because stages agree on their
interface, `a | b` just means "feed a's output to b". Prove it below entirely
offline with two toy runnables.

In [2]:
double = RunnableLambda(lambda n: n * 2)          # stage 1: any callable becomes a Runnable
label = RunnableLambda(lambda n: f"got {n}")      # stage 2: receives stage 1's output

pipeline = double | label                         # compose BEFORE ever invoking anything
print(pipeline.invoke(21))                        # single value -> single value
print(pipeline.batch([1, 2, 3]))                  # list in -> list out, same order
for piece in pipeline.stream(21):                 # lazy chunks: one per completed stage here
    print("chunk:", piece)

topic_prompt = ChatPromptTemplate.from_template("Tell me about {topic}.")   # templates too!
rendered = topic_prompt.invoke({"topic": "mars"}) # same verb, different component type
print(rendered.to_string()[:60], "...")           # preview exactly what a model would receive

got 42
['got 2', 'got 4', 'got 6']
chunk: got 42
Human: Tell me about mars. ...


## 2. Assembling prompt | model | parser, stage by stage

Build incrementally and inspect each prefix - never debug a whole pipeline blind.
The middle stage is swappable: a real `ChatGroq` when a key exists, otherwise an
offline echo model that still returns a genuine `AIMessage`. The surrounding chain
cannot tell the difference, which is precisely the point of the protocol.

In [3]:
quote_prompt = ChatPromptTemplate.from_template(
    "Give one famous quote by {person}, then attribute it."
)

stage1 = quote_prompt                             # step 1 alone: dict -> messages
print("stage 1:", repr(stage1.invoke({"person": "Grace Hopper"}).to_string()[:70]))

if not os.getenv("GROQ_API_KEY"):                 # mandatory guard before provider use
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env - using an offline echo model.")
    # OFFLINE branch consumes the raw DICT (skips the template) so types stay consistent:
    stage2 = RunnableLambda(lambda d: AIMessage(
        content=f"ECHO: {d['person']} once said something wise."))
else:
    from langchain_groq import ChatGroq
    stage2 = quote_prompt | ChatGroq(model="llama-3.3-70b-versatile",
                                     temperature=0)   # messages -> AIMessage
reply = stage2.invoke({"person": "Grace Hopper"})
print("stage 2:", type(reply).__name__, "|", repr(reply.content)[:60])

stage3 = stage2 | StrOutputParser()               # step 3: AIMessage -> plain str
final = stage3.invoke({"person": "Grace Hopper"})
print("stage 3:", type(final).__name__, "|", final[:80])

stage 1: 'Human: Give one famous quote by Grace Hopper, then attribute it.'
[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env - using an offline echo model.
stage 2: AIMessage | 'ECHO: Grace Hopper once said something wise.'
stage 3: TextAccessor | ECHO: Grace Hopper once said something wise.


## 3. RunnableLambda: glue for ordinary Python

Real pipelines need mundane code between model calls - unwrapping fields, cleaning
text, computing stats, routing decisions. Wrap any function in `RunnableLambda` and
it becomes a first-class stage: batchable, streamable, composable. Mid-chain is the
most common placement, hence the name you will hear: "glue".

In [4]:
def word_stats(text: str) -> str:
    """Toy transform: count words and find the longest - stands in for cleanup code."""
    words = text.replace(".", "").split()         # crude tokenization, good enough here
    longest = max(words, key=len)                 # tie goes to the earlier word
    return f"{len(words)} words; longest={longest}"


stats_chain = (
    RunnableLambda(lambda d: d["text"])           # unwrap the dict field FIRST...
    | RunnableLambda(word_stats)                  # ...then feed plain str to our function
)
print(stats_chain.invoke({"text": "LangChain lets plain functions join the pipe."}))

7 words; longest=LangChain


## 4. RunnablePassthrough + .assign(): keep everything, add more

A miniature preview of the RAG shape from the capstone. The user's question must
reach later stages UNCHANGED while retrieved context joins it. `RunnablePassthrough`
forwards the whole dict as-is; `.assign(new_key=fn)` returns a NEW runnable whose
output is the old dict PLUS the computed key. Chain several assigns to build up state.

> **Common pitfall:** the function inside `.assign()` receives the ENTIRE accumulated
> dict, not one field - read keys off `d`, and remember a same-name key would
> silently overwrite the original input.

In [5]:
def fake_retrieve(d: dict) -> str:
    """Stand-in for a real vector-store lookup later in the course."""
    return f"[doc snippet relevant to: {d['question']}]"


rag_shape = (
    RunnablePassthrough()                                  # forward {'question': ...} untouched
    .assign(context=fake_retrieve)                         # ADD context alongside question
    .assign(context_chars=lambda d: len(d["context"]))     # sees EVERYTHING assigned so far
)
snapshot = rag_shape.invoke({"question": "What is retrieval?"})
print(sorted(snapshot.keys()))                              # question AND context AND context_chars
print(snapshot["question"], "|", snapshot["context"])       # originals preserved, additions merged

['context', 'context_chars', 'question']
What is retrieval? | [doc snippet relevant to: What is retrieval?]


## 5. RunnableParallel: fan out, merge back

One input, several independent consumers, results merged into a dict keyed by
branch name. Below: a crude summary branch (first sentence) and a keywords branch
(sorted unique lowercase words) over the SAME text. A dict of runnables piped after
an input behaves identically to the explicit constructor shown here.

In [6]:
sample_text = {"text": "Chains compose. Small stages beat giant scripts. Test each prefix."}

summary_branch = RunnableLambda(
    lambda d: d["text"].split(".")[0].strip() + "."        # 'summary' = first sentence, cheaply
)
keyword_branch = RunnableLambda(
    lambda d: sorted({w.lower().strip(".") for w in d["text"].split()})[:5]   # top few, sorted
)

fanout = RunnableParallel(summary=summary_branch, keywords=keyword_branch)   # both see same input
merged = fanout.invoke(sample_text)
print(merged["summary"])                                    # branch one result
print(merged["keywords"])                                   # branch two result

if not os.getenv("GROQ_API_KEY"):                           # guard the LLM-flavored variant
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env for the live fan-out.")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    summarize = ChatPromptTemplate.from_template(
        "Summarize {topic} in one sentence."
    ) | llm | StrOutputParser()                              # branch A: prose
    keywords = ChatPromptTemplate.from_template(
        "List 4 comma separated keywords about {topic}."
    ) | llm | CommaSeparatedListOutputParser()               # branch B: real list

    llm_fanout = RunnableParallel(summary=summarize, keywords=keywords)
    print(llm_fanout.invoke({"topic": "solar power"}))       # merged dict of both branches

Chains compose.
['beat', 'chains', 'compose', 'each', 'giant']
[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env for the live fan-out.


## 6. Resilience: .with_fallbacks()

Providers rate-limit, model names vanish, networks hiccup. Wrapping a runnable with
fallbacks tries it first and transparently switches to backups IN ORDER when calls
fail. Here the primary points at a deliberately nonexistent model so you can watch
the safety net catch the fall.

> **Pro tip:** fallbacks swallow the root error, which makes incidents mysterious in
> production. Log before you fall back, and alert on HOW OFTEN the backup runs.

In [7]:
if not os.getenv("GROQ_API_KEY"):                            # guard: needs a valid key to work
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to watch fallbacks recover.")
else:
    from langchain_groq import ChatGroq

    flaky = ChatGroq(model="definitely-not-a-real-model")            # WILL fail at call time
    robust = flaky.with_fallbacks([ChatGroq(model="llama-3.3-70b-versatile")])
    answer = robust.invoke("Reply with exactly OK.")                 # primary dies, backup answers
    print("recovered via fallback:", answer.content)

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to watch fallbacks recover.


## 7. Streaming and batching

Two more protocol verbs doing real work:

- `stream()` yields tokens AS THE MODEL PRODUCES them - the trick behind typing-UX
  chat interfaces. Print chunks without newlines to watch text assemble live.
- `batch()` pushes several inputs through ONE composed chain; outputs come back as
  a list matching input order, so results zip safely onto their requests.

In [8]:
if not os.getenv("GROQ_API_KEY"):                            # guard the streaming cell
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to see token streaming.")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    explain = ChatPromptTemplate.from_template(
        "Explain {topic} in two short sentences."
    ) | llm | StrOutputParser()

    for token in explain.stream({"topic": "black holes"}):   # chunk-by-chunk arrival
        print(token, end="", flush=True)                     # no newline: tokens visibly stack
    print()                                                  # tidy newline after the burst

    topics = [{"topic": "gravity"}, {"topic": "comets"}]     # two requests, one chain
    for i, ans in enumerate(explain.batch(topics), start=1): # order matches input order
        print(f"[{i}]", ans[:70], "...")

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to see token streaming.


## 8. bind(): pin settings onto any model stage

`llm.bind(**kwargs)` returns a runnable with extra call-time arguments frozen inside -
same model, same chain position, different behavior. Common pins include
`temperature`, `tools` (agents, module ahead), and STOP SEQUENCES: generation halts
the moment output contains the stop string. That last one shines in ReAct-style
loops where the model must stop before hallucinating an `Observation:` line.

In [9]:
if not os.getenv("GROQ_API_KEY"):                            # guard the bound-model demo
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to try bind(stop=...).")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    bounded = llm.bind(stop=["."])                           # freeze a stop sequence inside
    truncated = bounded.invoke("Count from 1 to 10 using digits.")   # halts at first period
    print("bound output:", repr(truncated.content))          # ends early - no closing period

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to try bind(stop=...).


## Summary

- Every component answers to `invoke` / `batch` / `stream`; that uniformity IS LCEL.
- Build chains one stage at a time and inspect prefixes; swap live and offline
  middle stages freely because the protocol hides the difference.
- `RunnableLambda` brings ordinary Python into pipes; `RunnablePassthrough.assign`
  extends dicts in place - together they sketch the RAG shape before any retriever exists.
- `RunnableParallel` fans one input across branches and merges keyed results.
- `.with_fallbacks` adds a safety net (log the catches!), `.stream` buys UX, `.batch`
  buys throughput, and `bind(stop=...)` pins generation rules onto any model stage.

Next stop: memory - wrapping chains like today's with `RunnableWithMessageHistory`
so multi-turn conversation state rides along automatically.